# Lineage validation (Step 7, Part B)
Text-only lineage evidence for the FE Bar build. Path checked:
`/Volumes/.../staging` -> bronze -> silver -> `txn_features_gold` / `customer_risk_profile_gold` -> `scored_transactions` -> synced tables in `fe_bar_lakebase.fraud.*`.

Sources, in order of preference:
1. `system.access.table_lineage` and `system.access.column_lineage` (UC-captured lineage).
2. Hops that system tables do not capture are filled with the next-best text evidence (pipeline definition, synced-table spec) and labelled **NEXT-BEST EVIDENCE**.
3. Model lineage from Unity Catalog model registry + MLflow run inputs.

Read-only. Nothing is written.

In [0]:
# Step 7 Part B: text lineage evidence (replaces the lineage-graph screenshot in the build plan).
# Read-only: queries system.access.* lineage, pipeline/synced-table definitions and MLflow. Writes nothing.
import datetime, json
UTC = datetime.timezone.utc
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

CAT, SCH = "workshop_guy_catalog", "fe_bar_a0"
GOLD_PIPELINE = "c0db28b8-dbd2-432d-99cf-4ea2aea60274"
SYNC_PIPELINE = "f8690359-4bb1-45b9-8fdb-5bd892239db2"
JOB_ID = 1115396029932218
REPO = "/Workspace/Users/alex.owen@databricks.com/FE-BAR-2026-repo"
SINCE = "2026-10-01"
print("run_at_utc:", datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"))
print("current_user:", spark.sql("SELECT current_user()").first()[0])

def md(rows, cols):
    """Print rows (list of dicts) as a GFM markdown table so the evidence is readable as text."""
    print("| " + " | ".join(cols) + " |")
    print("|" + "|".join(["---"] * len(cols)) + "|")
    for r in rows:
        print("| " + " | ".join("" if r.get(c) is None else str(r.get(c)) for c in cols) + " |")
    print(f"({len(rows)} rows)")

# Map entity IDs to readable names: repo notebooks by object ID, plus the pipelines and job.
ENTITY = {GOLD_PIPELINE: "pipeline: FE BAR Bronze Silver AutoLoader Pipeline",
          SYNC_PIPELINE: "pipeline: Lakebase sync (SNAPSHOT)",
          str(JOB_ID): "job: fe_bar_gold_to_lakebase"}
def _walk(p):
    for o in w.workspace.list(p):
        if o.object_type.value == "DIRECTORY":
            yield from _walk(o.path)
        elif o.object_type.value == "NOTEBOOK":
            yield o
for o in _walk(REPO):
    ENTITY[str(o.object_id)] = "notebook: " + o.path.replace(REPO + "/", "")
# Earlier object IDs of the same notebooks (objects were re-created by Git checkouts / moves), and
# Genie Code conversation execution contexts (the conversation ID is the context's notebook object ID).
ENTITY.update({
    "2565515039400054": "notebook: fraud_model_training (Step 4 object id)",
    "2565515039400055": "notebook: fraud_batch_scoring (Step 4 object id)",
    "3458967129203279": "notebook: card_transaction_bronze_generator (Step 1 object id, old fe-bar-a0 schema)",
    "1835980093459164": "notebook: dashboard_validation (Step 6 object id)",
    "1853835927236788": "notebook: fgac_hardening (Step 5 object id)",
    "2606072674702398": "notebook: gold_feature_validation (Step 2 object id)",
})
for conv, step in {"3458967129203277": "Step 1", "2172058462486781": "Step 2", "2606072674702393": "Step 2 gold",
                   "3367633421784666": "Step 3", "2565515039400048": "Step 4", "1507503687185838": "Step 5",
                   "1835980093459154": "Step 6", "245211859776391": "Step 7"}.items():
    ENTITY.setdefault(conv, f"Genie conversation {conv} ({step})")

def in_scope(name):
    """Exact prefix match (LIKE treats '_' as a wildcard and would also match the old fe-bar-a0 schema)."""
    return bool(name) and (name.startswith(f"{CAT}.{SCH}.") or name.startswith("fe_bar_lakebase."))
def ent(t, i):
    return ENTITY.get(str(i), f"{(t or 'NULL').lower()}: {i}") if i else (t or "NULL")
print(f"{len(ENTITY)} entity names resolved")

run_at_utc: 2026-11-01T19:04:38+00:00
current_user: alex.owen@databricks.com
30 entity names resolved


In [0]:
# One row per (source -> target, entity) edge for the FE Bar objects, with the latest event time.
FQ = f"{CAT}.{SCH}"
edges_df = spark.sql(f"""
  SELECT coalesce(source_table_full_name, source_path) AS source, source_type,
         target_table_full_name AS target, target_type,
         entity_type, entity_id, count(*) AS events, max(event_time) AS latest_event_utc
  FROM system.access.table_lineage
  WHERE event_date >= '{SINCE}'
    AND target_table_full_name IS NOT NULL
    AND (source_table_full_name IS NOT NULL OR source_path IS NOT NULL)
    AND (target_table_full_name LIKE '{FQ}.%' OR target_table_full_name LIKE 'fe_bar_lakebase.%')
    AND target_table_full_name NOT LIKE '{FQ}.event_log_%'
  GROUP BY ALL
  ORDER BY latest_event_utc
""")
EDGES = [r.asDict() for r in edges_df.collect() if in_scope(r["target"])]
for r in EDGES:
    r["entity"] = ent(r["entity_type"], r["entity_id"])
    r["latest_event_utc"] = str(r["latest_event_utc"])[:19]
md(EDGES, ["source", "source_type", "target", "target_type", "entity_type", "entity", "events", "latest_event_utc"])

| source | source_type | target | target_type | entity_type | entity | events | latest_event_utc |
|---|---|---|---|---|---|---|---|
| workshop_guy_catalog.fe_bar_a0.dim_customer_silver | TABLE | workshop_guy_catalog.fe_bar_a0.customer_risk_profile_gold | MATERIALIZED_VIEW | PIPELINE | pipeline: FE BAR Bronze Silver AutoLoader Pipeline | 3 | 2026-10-17 01:15:17 |
| workshop_guy_catalog.fe_bar_a0.txn_features_gold | TABLE | workshop_guy_catalog.fe_bar_a0.scored_transactions | TABLE | NOTEBOOK | notebook: fraud_batch_scoring (Step 4 object id) | 1 | 2026-10-20 02:41:01 |
| workshop_guy_catalog.fe_bar_a0.scored_transactions | TABLE | fe_bar_lakebase.fraud.txn_risk_current | TABLE | PIPELINE | pipeline: Lakebase sync (SNAPSHOT) | 1 | 2026-10-20 04:25:36 |
| workshop_guy_catalog.fe_bar_a0.dim_merchant_bronze | STREAMING_TABLE | workshop_guy_catalog.fe_bar_a0.dim_merchant_silver | MATERIALIZED_VIEW | PIPELINE | pipeline: FE BAR Bronze Silver AutoLoader Pipeline | 5 | 2026-10-21 02:34:26 |
| 

In [0]:
# Expected hops of the FE Bar path. FOUND = at least one edge in system.access.table_lineage.
VOL = f"/Volumes/{CAT}/{SCH}/staging"
EXPECTED = [
  ("1 raw->bronze", f"{VOL}/transactions", f"{FQ}.fact_transactions_bronze"),
  ("1 raw->bronze", f"{VOL}/customers", f"{FQ}.dim_customer_bronze"),
  ("1 raw->bronze", f"{VOL}/merchants", f"{FQ}.dim_merchant_bronze"),
  ("1 raw->bronze", f"{VOL}/prior_fraud_cases", f"{FQ}.fact_prior_fraud_cases_bronze"),
  ("2 bronze->silver", f"{FQ}.fact_transactions_bronze", f"{FQ}.fact_transactions_silver"),
  ("2 bronze->silver", f"{FQ}.dim_customer_bronze", f"{FQ}.dim_customer_silver"),
  ("2 bronze->silver", f"{FQ}.dim_merchant_bronze", f"{FQ}.dim_merchant_silver"),
  ("2 bronze->silver", f"{FQ}.fact_prior_fraud_cases_bronze", f"{FQ}.fact_prior_fraud_cases_silver"),
  ("3 silver->gold", f"{FQ}.fact_transactions_silver", f"{FQ}.txn_features_gold"),
  ("3 silver->gold", f"{FQ}.dim_customer_silver", f"{FQ}.txn_features_gold"),
  ("3 silver->gold", f"{FQ}.dim_merchant_silver", f"{FQ}.txn_features_gold"),
  ("3 silver->gold", f"{FQ}.fact_transactions_silver", f"{FQ}.customer_risk_profile_gold"),
  ("3 silver->gold", f"{FQ}.dim_customer_silver", f"{FQ}.customer_risk_profile_gold"),
  ("3 silver->gold", f"{FQ}.fact_prior_fraud_cases_silver", f"{FQ}.customer_risk_profile_gold"),
  ("4 gold->score", f"{FQ}.txn_features_gold", f"{FQ}.scored_transactions"),
  ("5 gold->lakebase", f"{FQ}.txn_features_gold", "fe_bar_lakebase.fraud.txn_features"),
  ("5 gold->lakebase", f"{FQ}.customer_risk_profile_gold", "fe_bar_lakebase.fraud.customer_risk_profile"),
  ("5 score->lakebase", f"{FQ}.scored_transactions", "fe_bar_lakebase.fraud.txn_risk_current"),
  ("5 calib->lakebase", f"{FQ}.risk_score_calibration", "fe_bar_lakebase.fraud.risk_score_calibration"),
]
HOPS, MISSING = [], []
for hop, src, tgt in EXPECTED:
    m = [e for e in EDGES if e["target"] == tgt and (e["source"] == src or str(e["source"]).startswith(src))]
    found = bool(m)
    HOPS.append({"hop": hop, "source": src, "target": tgt,
                 "in_system_tables": "FOUND" if found else "MISSING",
                 "entity": ", ".join(sorted({e["entity"] for e in m})) if m else "",
                 "latest_event_utc": max(e["latest_event_utc"] for e in m) if m else ""})
    if not found:
        MISSING.append((hop, src, tgt))
md(HOPS, ["hop", "source", "target", "in_system_tables", "entity", "latest_event_utc"])
print(f"\nFOUND {len(EXPECTED)-len(MISSING)}/{len(EXPECTED)} hops in system.access.table_lineage; MISSING {len(MISSING)}:")
for h in MISSING:
    print("  MISSING:", h[0], h[1], "->", h[2])

| hop | source | target | in_system_tables | entity | latest_event_utc |
|---|---|---|---|---|---|
| 1 raw->bronze | /Volumes/workshop_guy_catalog/fe_bar_a0/staging/transactions | workshop_guy_catalog.fe_bar_a0.fact_transactions_bronze | MISSING |  |  |
| 1 raw->bronze | /Volumes/workshop_guy_catalog/fe_bar_a0/staging/customers | workshop_guy_catalog.fe_bar_a0.dim_customer_bronze | MISSING |  |  |
| 1 raw->bronze | /Volumes/workshop_guy_catalog/fe_bar_a0/staging/merchants | workshop_guy_catalog.fe_bar_a0.dim_merchant_bronze | MISSING |  |  |
| 1 raw->bronze | /Volumes/workshop_guy_catalog/fe_bar_a0/staging/prior_fraud_cases | workshop_guy_catalog.fe_bar_a0.fact_prior_fraud_cases_bronze | MISSING |  |  |
| 2 bronze->silver | workshop_guy_catalog.fe_bar_a0.fact_transactions_bronze | workshop_guy_catalog.fe_bar_a0.fact_transactions_silver | FOUND | pipeline: FE BAR Bronze Silver AutoLoader Pipeline | 2026-10-21 02:34:26 |
| 2 bronze->silver | workshop_guy_catalog.fe_bar_a0.dim_customer_br

In [0]:
# Labelled NEXT-BEST EVIDENCE: definitions that declare source -> target, used only for hops B.2 marked MISSING.
import re
print("=== NEXT-BEST EVIDENCE (a): SDP pipeline definition, raw volume -> bronze ===")
pl = w.pipelines.get(GOLD_PIPELINE)
print(f"pipeline {GOLD_PIPELINE} '{pl.name}' catalog={pl.spec.catalog} schema={pl.spec.schema or pl.spec.target} serverless={pl.spec.serverless}")
libs = []
for lib in pl.spec.libraries or []:
    p = (lib.notebook.path if lib.notebook else None) or (lib.file.path if lib.file else None)
    libs.append(p); print("  library:", p)
rows = []
for p in libs:
    try:
        src = open("/Workspace" + p if not p.startswith("/Workspace") else p).read()
    except Exception:
        src = open("/Workspace" + p + ".sql").read() if not p.endswith(".sql") else ""
    # Each bronze streaming table and the volume path it reads with read_files/cloud_files.
    for m in re.finditer(r"CREATE\s+(?:OR\s+REFRESH\s+)?STREAMING\s+TABLE\s+([\w.`]+)(.*?)(?=CREATE\s+(?:OR\s+REFRESH\s+)?(?:STREAMING\s+TABLE|MATERIALIZED\s+VIEW)|\Z)", src, re.S | re.I):
        paths = re.findall(r"['\"](/Volumes/[^'\"]+)['\"]", m.group(2))
        rows.append({"bronze_table": m.group(1).strip("`"), "reads_volume_path": ", ".join(paths) or "(none found)",
                     "defined_in": p.split("/")[-1], "evidence": "NEXT-BEST: pipeline source"})
md(rows, ["bronze_table", "reads_volume_path", "defined_in", "evidence"])
last = [u for u in (pl.latest_updates or [])][:1]
for u in last:
    print(f"latest pipeline update {u.update_id} state={u.state.value} created={u.creation_time}")

print("\n=== NEXT-BEST EVIDENCE (b): Lakebase synced-table specs (source -> target) ===")
srows = []
for n in ["fe_bar_lakebase.fraud.txn_features", "fe_bar_lakebase.fraud.customer_risk_profile",
          "fe_bar_lakebase.fraud.txn_risk_current", "fe_bar_lakebase.fraud.risk_score_calibration"]:
    st = w.database.get_synced_database_table(n)
    ds = st.data_synchronization_status
    srows.append({"synced_table": n, "source_table": st.spec.source_table_full_name,
                  "primary_key": ",".join(st.spec.primary_key_columns or []),
                  "policy": st.spec.scheduling_policy.value, "pipeline_id": ds.pipeline_id if ds else None,
                  "state": ds.detailed_state.value if ds else None,
                  "in_table_lineage": "FOUND" if any(e["target"] == n for e in EDGES) else "MISSING (spec is evidence)"})
md(srows, ["synced_table", "source_table", "primary_key", "policy", "pipeline_id", "state", "in_table_lineage"])
sp = w.pipelines.get(SYNC_PIPELINE)
for u in (sp.latest_updates or [])[:1]:
    print(f"sync pipeline {SYNC_PIPELINE} latest update {u.update_id} state={u.state.value} created={u.creation_time}")

=== NEXT-BEST EVIDENCE (a): SDP pipeline definition, raw volume -> bronze ===
pipeline c0db28b8-dbd2-432d-99cf-4ea2aea60274 'FE BAR Bronze Silver AutoLoader Pipeline' catalog=workshop_guy_catalog schema=fe_bar_a0 serverless=True
  library: /Workspace/Users/alex.owen@databricks.com/FE-BAR-2026-repo/FE BAR Bronze Silver AutoLoader Pipeline
  library: /Workspace/Users/alex.owen@databricks.com/FE-BAR-2026-repo/gold_feature_layer.sql
| bronze_table | reads_volume_path | defined_in | evidence |
|---|---|---|---|
| dim_customer_bronze | /Volumes/workshop_guy_catalog/fe_bar_a0/staging/customers/ | FE BAR Bronze Silver AutoLoader Pipeline | NEXT-BEST: pipeline source |
| dim_merchant_bronze | /Volumes/workshop_guy_catalog/fe_bar_a0/staging/merchants/ | FE BAR Bronze Silver AutoLoader Pipeline | NEXT-BEST: pipeline source |
| fact_transactions_bronze | /Volumes/workshop_guy_catalog/fe_bar_a0/staging/transactions/ | FE BAR Bronze Silver AutoLoader Pipeline | NEXT-BEST: pipeline source |
| fact_pr

In [0]:
# Governed columns: dim_customer_silver.avg_monthly_spend (column mask), fact_transactions_silver.amount (row filter > 10K),
# and the derived txn_features_gold.amt_to_monthly_spend (masked in fraud_gov views).
col_df = spark.sql(f"""
  SELECT source_table_full_name AS source_table, source_column_name AS source_column,
         coalesce(target_table_full_name, '(read only)') AS target_table, coalesce(target_column_name, '') AS target_column,
         entity_type, entity_id, count(*) AS events, max(event_time) AS latest_event_utc
  FROM system.access.column_lineage
  WHERE event_date >= '{SINCE}'
    AND (source_table_full_name LIKE '{FQ}.%' OR source_table_full_name LIKE 'fe_bar_lakebase.%')
    AND source_column_name IN ('avg_monthly_spend', 'amount', 'amt_to_monthly_spend')
  GROUP BY ALL
  ORDER BY source_table, source_column, latest_event_utc
""")
COLS = [r.asDict() for r in col_df.collect() if in_scope(r["source_table"])]
for r in COLS:
    r["entity"] = ent(r["entity_type"], r["entity_id"]); r["latest_event_utc"] = str(r["latest_event_utc"])[:19]
with_target = [r for r in COLS if r["target_table"] != "(read only)"]
print("=== (a) Column-to-column edges for governed columns ===")
md(with_target, ["source_table", "source_column", "target_table", "target_column", "entity", "events", "latest_event_utc"])
# (b) Reads (no target column): one row per governed column and entity type, with the readers listed.
agg = {}
for r in COLS:
    if r["target_table"] != "(read only)":
        continue
    k = (r["source_table"], r["source_column"], r["entity_type"] or "NULL")
    a = agg.setdefault(k, {"source_table": k[0], "source_column": k[1], "entity_type": k[2], "events": 0, "readers": set(), "latest_event_utc": ""})
    a["events"] += r["events"]; a["readers"].add(r["entity"]); a["latest_event_utc"] = max(a["latest_event_utc"], r["latest_event_utc"])
reads = sorted(agg.values(), key=lambda a: (a["source_table"], a["source_column"], a["entity_type"]))
for a in reads:
    a["readers"] = "; ".join(sorted(a["readers"]))
print("\n=== (b) Reads of governed columns (no target column), grouped by entity type ===")
md(reads, ["source_table", "source_column", "entity_type", "events", "readers", "latest_event_utc"])
print(f"\n{len(COLS)} column-lineage rows; {len(with_target)} are column-to-column edges.")
for t, c in [(f"{FQ}.dim_customer_silver", "avg_monthly_spend"), (f"{FQ}.fact_transactions_silver", "amount"), (f"{FQ}.txn_features_gold", "amt_to_monthly_spend")]:
    print(f"  {t}.{c}: {'present' if any(r['source_table']==t and r['source_column']==c for r in COLS) else 'NOT in column_lineage'}")

=== (a) Column-to-column edges for governed columns ===
| source_table | source_column | target_table | target_column | entity | events | latest_event_utc |
|---|---|---|---|---|---|---|
| workshop_guy_catalog.fe_bar_a0.dim_customer_silver | avg_monthly_spend | workshop_guy_catalog.fe_bar_a0.txn_features_gold | amt_to_monthly_spend | pipeline: FE BAR Bronze Silver AutoLoader Pipeline | 4 | 2026-10-17 02:27:13 |
| workshop_guy_catalog.fe_bar_a0.fact_transactions_silver | amount | workshop_guy_catalog.fe_bar_a0.txn_features_gold | __mv__expectations | pipeline: FE BAR Bronze Silver AutoLoader Pipeline | 4 | 2026-10-17 02:27:13 |
| workshop_guy_catalog.fe_bar_a0.fact_transactions_silver | amount | workshop_guy_catalog.fe_bar_a0.txn_features_gold | amount | pipeline: FE BAR Bronze Silver AutoLoader Pipeline | 4 | 2026-10-17 02:27:13 |
| workshop_guy_catalog.fe_bar_a0.fact_transactions_silver | amount | workshop_guy_catalog.fe_bar_a0.txn_features_gold | amt_to_cust_avg_30d | pipeline: FE BA

In [0]:
import mlflow
from mlflow.tracking import MlflowClient
mlflow.set_registry_uri("databricks-uc")
mc = MlflowClient(registry_uri="databricks-uc")
MODEL = f"{FQ}.fraud_risk_model"
mv = mc.get_model_version(MODEL, "1")
champ = mc.get_model_version_by_alias(MODEL, "champion")
run = mc.get_run(mv.run_id)
print(f"model {MODEL} v{mv.version} (champion -> v{champ.version}) run_id={mv.run_id}")
print(f"  run name={run.info.run_name} start={datetime.datetime.fromtimestamp(run.info.start_time/1000, UTC).isoformat(timespec='seconds')}")
print(f"  source notebook={run.data.tags.get('mlflow.source.name')} (id {run.data.tags.get('mlflow.databricks.notebookID')})")
m = run.data.metrics
print(f"  metrics: " + ", ".join(f"{k}={v:.4f}" for k, v in sorted(m.items()) if k in ('test_roc_auc','test_pr_auc','roc_auc','pr_auc')))
print(f"  params: block_threshold={run.data.params.get('block_threshold')} review_threshold={run.data.params.get('review_threshold')}")

print("\n=== MLflow logged dataset inputs (run.inputs.dataset_inputs) ===")
drows = []
for di in run.inputs.dataset_inputs:
    d = di.dataset
    prof = json.loads(d.profile) if d.profile else {}
    src = json.loads(d.source) if d.source else {}
    drows.append({"context": ",".join(t.value for t in di.tags if t.key == "mlflow.data.context"),
                  "digest": d.digest, "source_type": d.source_type, "num_rows": prof.get("num_rows"),
                  "source": src.get("uri") or src.get("table_name") or src.get("tags", {}).get("mlflow.source.name")})
md(drows, ["context", "digest", "source_type", "num_rows", "source"])
print("Note: datasets were logged from in-memory pandas frames (source_type=code), so the MLflow run does not name a UC table.")
print("The UC table the training notebook read is taken from system.access.table_lineage below.")

nb_id = run.data.tags.get("mlflow.databricks.notebookID")
tr = [r.asDict() for r in spark.sql(f"""
  SELECT source_table_full_name AS source, entity_type, entity_id, count(*) AS events, min(event_time) AS first_utc, max(event_time) AS latest_utc
  FROM system.access.table_lineage
  WHERE event_date >= '{SINCE}' AND entity_id = '{nb_id}' AND source_table_full_name IS NOT NULL
  GROUP BY ALL ORDER BY events DESC""").collect()]
for r in tr:
    r["entity"] = ent(r["entity_type"], r["entity_id"]); r["first_utc"] = str(r["first_utc"])[:19]; r["latest_utc"] = str(r["latest_utc"])[:19]
print(f"\n=== Tables read by the training notebook (entity_id {nb_id}) ===")
md(tr, ["source", "entity", "events", "first_utc", "latest_utc"])

print("\n=== Writers of scored_transactions (system.access.table_lineage) ===")
wr = [r.asDict() for r in spark.sql(f"""
  SELECT entity_type, entity_id, entity_run_id, max(event_time) AS latest_utc
  FROM system.access.table_lineage
  WHERE event_date >= '{SINCE}' AND target_table_full_name = '{FQ}.scored_transactions'
  GROUP BY ALL ORDER BY latest_utc""").collect()]
for r in wr:
    r["entity"] = ent(r["entity_type"], r["entity_id"]); r["latest_utc"] = str(r["latest_utc"])[:19]
md(wr, ["entity_type", "entity", "entity_run_id", "latest_utc"])
job = w.jobs.get(JOB_ID)
print(f"\njob {JOB_ID} '{job.settings.name}' tasks: " + " -> ".join(t.task_key for t in job.settings.tasks))
for t in job.settings.tasks:
    tgt = (t.notebook_task.notebook_path if t.notebook_task else None) or (t.pipeline_task.pipeline_id if t.pipeline_task else None)
    print(f"  task {t.task_key}: {tgt}")

model workshop_guy_catalog.fe_bar_a0.fraud_risk_model v1 (champion -> v1) run_id=d89beaca8f81447780da60ed2ea7c811
  run name=lgbm_fraud_20261020_023229 start=2026-10-20T02:32:29+00:00
  source notebook=/Users/alex.owen@databricks.com/FE-BAR-2026-repo/fraud_model_training (id 2565515039400054)
  metrics: test_pr_auc=0.2808, test_roc_auc=0.8448
  params: block_threshold=0.9502674433961342 review_threshold=0.8714158290727556

=== MLflow logged dataset inputs (run.inputs.dataset_inputs) ===
| context | digest | source_type | num_rows | source |
|---|---|---|---|---|
| eval | e8bde978 | code | 92927 | /Users/alex.owen@databricks.com/FE-BAR-2026-repo/fraud_model_training |
| train | 9e09afd8 | code | 371705 | /Users/alex.owen@databricks.com/FE-BAR-2026-repo/fraud_model_training |
(2 rows)
Note: datasets were logged from in-memory pandas frames (source_type=code), so the MLflow run does not name a UC table.
The UC table the training notebook read is taken from system.access.table_lineage belo

In [0]:
print("=== Lineage summary ===")
print(f"table_lineage edges for FE Bar objects: {len(EDGES)}")
print(f"expected hops found in system tables: {len(EXPECTED)-len(MISSING)}/{len(EXPECTED)}")
for h in MISSING:
    print(f"  MISSING from system tables (filled by NEXT-BEST EVIDENCE in B.3): {h[0]}  {h[1]} -> {h[2]}")
print(f"column_lineage rows for governed columns: {len(COLS)}")
print(f"model {MODEL} v{mv.version} <- run {mv.run_id}; training notebook read: {', '.join(r['source'] for r in tr) or 'none captured'}")
print(f"scored_transactions writers: {', '.join(sorted({r['entity'] for r in wr}))}")

=== Lineage summary ===
table_lineage edges for FE Bar objects: 18
expected hops found in system tables: 13/19
  MISSING from system tables (filled by NEXT-BEST EVIDENCE in B.3): 1 raw->bronze  /Volumes/workshop_guy_catalog/fe_bar_a0/staging/transactions -> workshop_guy_catalog.fe_bar_a0.fact_transactions_bronze
  MISSING from system tables (filled by NEXT-BEST EVIDENCE in B.3): 1 raw->bronze  /Volumes/workshop_guy_catalog/fe_bar_a0/staging/customers -> workshop_guy_catalog.fe_bar_a0.dim_customer_bronze
  MISSING from system tables (filled by NEXT-BEST EVIDENCE in B.3): 1 raw->bronze  /Volumes/workshop_guy_catalog/fe_bar_a0/staging/merchants -> workshop_guy_catalog.fe_bar_a0.dim_merchant_bronze
  MISSING from system tables (filled by NEXT-BEST EVIDENCE in B.3): 1 raw->bronze  /Volumes/workshop_guy_catalog/fe_bar_a0/staging/prior_fraud_cases -> workshop_guy_catalog.fe_bar_a0.fact_prior_fraud_cases_bronze
  MISSING from system tables (filled by NEXT-BEST EVIDENCE in B.3): 5 gold->lakebas